# 1. Library
---

In [ ]:
import pandas as pd
import requests
from datetime import datetime
import time

# SQL 
from sqlalchemy import create_engine, Table, Column, Integer, String, Float, DateTime, MetaData, ForeignKey, delete, text

# Access
import os
from dotenv import load_dotenv
load_dotenv()


# 2. Data
---

In [ ]:
list_cities = ["Le Mont Saint Michel","St Malo","Bayeux","Le Havre","Rouen","Paris","Amiens","Lille","Strasbourg","Chateau du Haut Koenigsbourg","Colmar","Eguisheim","Besancon","Dijon","Annecy","Grenoble","Lyon","Moustiers-Sainte-Marie","Bormes les Mimosas","Cassis","Marseille","Aix en Provence","Avignon","Uzes","Nimes","Aigues Mortes","Saintes Maries de la mer","Collioure","Carcassonne","Tarascon sur Ariege","Toulouse","Montauban","Biarritz","Bayonne","La Rochelle"]
list_cities = sorted(list_cities)

# 3. ETL
---

In [ ]:
# Credentials
database = os.environ["POSTGRES_DATABASE"]

#start engine
engine = create_engine(database)

# Connect to the db
conn = engine.connect()

## 3.1 Cities : Geolocation with Nominatim

In [ ]:
url = "https://nominatim.openstreetmap.org/search"

df_cities = pd.DataFrame()
i = 0

for city in list_cities:
    i += 1

    params = {
        "q": f"{city}, France",
        "format": "json",
        "limit": 1
    }

    headers = {
        "User-Agent": "trip-reco-project (jean@bon.beurre)"
    }

    try:
        response = requests.get(url, params=params, headers=headers, timeout=10)

        if response.status_code == 429:
            print(f"{city}  Rate limited (429)")
            break

        response.raise_for_status()

        data_json = response.json()

        if not data_json:
            print(f"{city}  Not found")
            continue

        city_data = {
            "city_id": i,
            "name": data_json[0]["name"],
            "latitude": float(data_json[0]["lat"]),
            "longitude": float(data_json[0]["lon"])
        }

        df_cities = pd.concat(
            [df_cities, pd.DataFrame([city_data])],
            ignore_index=True
        )

        print(f"{city}  done")

        time.sleep(1.2)  # respect rate limit

    except requests.exceptions.RequestException as e:
        print(f"{city}  Request error:", e)
        continue
    


In [ ]:
# load data
df_cities.to_sql(
    "trip_reco_cities",
    engine,
    if_exists="append",
    index=False
)

## 3.2 Weather with OpenWeather API

In [ ]:

url = "https://api.openweathermap.org/data/2.5/forecast?"

df_weather = pd.DataFrame()
for i in range(len(df_cities)):

    params = {
        'lat': df_cities.loc[i,'latitude'],
        'lon': df_cities.loc[i,'longitude'],
        'units': 'metric',
        'appid': os.environ["OPENWEATHERMAP_API_KEY"]}
    
    
    req = requests.get(url, params=params).json()

    for dt in req['list']:

        data = {
            'city_id' : df_cities.loc[i,'city_id'],
            'datetime' : datetime.fromtimestamp(dt['dt']),
            'temp_min' : dt['main']['temp_min'],
            'temp_max' : dt['main']['temp_max'],
            'temp_feels_like' : dt['main']['feels_like'],
            'sky': dt['weather'][0]['description']
            }
    
        data= pd.DataFrame([data])
        int(datetime.timestamp(datetime.now()) * 1000)

        df_weather = pd.concat([df_weather, data],ignore_index=True)
    print(df_cities.loc[i,'name']," done")
    
df_weather['updated_at'] = int(datetime.timestamp(datetime.now()) * 1000)

In [ ]:
# load data
df_weather.to_sql(
    "trip_reco_weathers",
    engine,
    if_exists="append",
    index=False
)

## 3.3 Hotel from CSV


In [ ]:
df_hotels = pd.read_csv('../data/hotels.csv', index_col='hotel_id')

In [ ]:
df_hotels.head()

In [ ]:
df_hotels.to_sql("trip_reco_hotels", engine, if_exists = "replace")